# DistilBERT for Fake News Detection / DistilBERT 假新闻检测

This notebook fine-tunes DistilBERT on the same debiased dataset used in `02_lstm_corrected.ipynb`, enabling a fair comparison between the optimised LSTM and a Transformer-based model.

本 notebook 在与 `02_lstm_corrected.ipynb` 完全相同的去偏数据上微调 DistilBERT，目的是在同等条件下对比优化版 LSTM 与 Transformer 模型的表现。

---

**为什么选 DistilBERT？/ Why DistilBERT?**

DistilBERT 是 BERT 的轻量蒸馏版本，体积缩小 40%，速度提升 60%，同时保留了 97% 的语言理解能力。与 LSTM 从零学习词向量不同，DistilBERT 带有在大规模语料上预训练好的上下文词向量，对语言有更深层的理解——这正是处理真假新闻细微写作风格差异所需要的能力。

DistilBERT is a distilled version of BERT — 40% smaller and 60% faster, retaining 97% of BERT's language understanding. Unlike LSTM which learns word representations from scratch, DistilBERT brings pretrained contextual embeddings, giving it a fundamentally richer understanding of language — exactly what is needed to capture subtle stylistic differences between real and fake news.

## 1. Imports / 导入库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import html as html_module

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import get_scheduler
from torch.optim import AdamW

os.makedirs('../results', exist_ok=True)

# 优先使用 Apple Silicon GPU（MPS），其次 CUDA，最后 CPU
# Use Apple Silicon GPU (MPS) if available, then CUDA, then CPU
device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 2. Load & Clean Data / 加载与清洗数据

与 `02_lstm_corrected.ipynb` 完全相同的九步清洗流程，确保两个模型在同一干净数据上对比。

Identical nine-step cleaning pipeline as `02_lstm_corrected.ipynb` — ensures any difference in results is attributable to model architecture, not data.

In [ ]:
fake = pd.read_csv('../data/raw/Fake.csv')
true = pd.read_csv('../data/raw/True.csv')
fake['label'] = 0
true['label'] = 1
df = pd.concat([fake, true]).sample(frac=1, random_state=42).reset_index(drop=True)

df = df.drop(columns=['subject', 'date'])

df['text'] = df['text'].str.replace(r'[\w\s]+\(Reuters\)\s*-?\s*', '', regex=True)
df['text'] = df['text'].str.replace(r'Reuters', '', regex=False)
df['text'] = df['text'].str.replace(r'^\s*[A-Z][A-Z\s\.]+\s*-\s*', '', regex=True)
df['text'] = df['text'].str.replace(r'[\w\s]+\((AP|AFP|UPI)\)\s*-?\s*', '', regex=True)
df['text'] = df['text'].str.replace(r'\b(AP|AFP|UPI)\b', '', regex=True)
df['text'] = df['text'].str.replace(r'http\S+|www\.\S+', '', regex=True)
df['text'] = df['text'].str.replace(r'@\w+', '', regex=True)
df['text'] = df['text'].apply(html_module.unescape)
df['text'] = df['text'].str.replace(r'[^\x00-\x7F]+', ' ', regex=True)
df['text'] = df['text'].str.strip()
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True)

df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
before = len(df)
df = df[df['content'].str.split().str.len() > 10].reset_index(drop=True)

print(f'Total after cleaning: {len(df):,} rows ({before - len(df)} removed)')
df[['content', 'label']].head(3)

清洗流程与 `02_lstm_corrected.ipynb` 完全一致，两个模型看到的是同一份数据，结果的差异只能来自模型架构本身。

Cleaning is identical to `02_lstm_corrected.ipynb`. Any performance difference between this model and the optimised LSTM reflects the architecture, not the data.